In [ ]:
import os
import re
import gc
import time
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import google.generativeai as genai
import warnings
from dotenv import load_dotenv

# Suppress PearsonR constant input warnings for dead features
warnings.filterwarnings("ignore", category=UserWarning, module="scipy.stats")

# ==========================================
# 1. Configuration & Paths
# ==========================================
# Load local .env file if it exists
load_dotenv()
try:
    # Attempt to load from Kaggle Secrets
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("GEMINI_API_KEY")
except ImportError:
    # Fallback to local environment variable
    api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY not found. Set it in Kaggle Secrets or a local .env file.")

genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.5-flash') 

# Detect environment
IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    BASE_ACT_PATH = "/kaggle/input/datasets/shaurya01pratap/saellama3/"
    BASE_DATA_PATH = "/kaggle/input/datasets/shaurya01pratap/principledata/New Clean Data/"
    OUTPUT_PATH = "/kaggle/working/"
else:
    # Local paths (assumes users download data into a 'data' folder at the repo root)
    BASE_ACT_PATH = "../data/saellama3/"
    BASE_DATA_PATH = "../data/principledata/New Clean Data/"
    OUTPUT_PATH = "../output/"
    
    # Ensure local output directory exists
    os.makedirs(OUTPUT_PATH, exist_ok=True)

LAYER_MAPPING = {
    1: 17, 2: 14, 3: 15, 4: 4, 5: 16, 6: 19,
    7: 28, 8: 3, 9: 13, 10: 11, 11: 9, 12: 13
}

# ==========================================
# 2. Core Functions
# ==========================================

def safe_generate(prompt_text, max_retries=6, initial_delay=5):
    delay = initial_delay
    for attempt in range(max_retries):
        try:
            response = model.generate_content(prompt_text)
            return response.text.strip()
        except Exception as e:
            error_msg = str(e).lower()
            if "429" in error_msg or "quota" in error_msg or "exhausted" in error_msg:
                print(f"         [!] Rate limit hit. Retrying in {delay}s... (Attempt {attempt+1}/{max_retries})")
                time.sleep(delay)
                delay *= 2  
            else:
                print(f"         [!] Non-recoverable API error: {e}")
                return None
    print("         [!] Max retries exceeded. Skipping to prevent crash.")
    return None

def load_data(principle_idx, layer_idx):
    act_file = f"{BASE_ACT_PATH}Raw_Activations_Principle{principle_idx}_Layer_{layer_idx}.csv"
    prompt_file = f"{BASE_DATA_PATH}Principle{principle_idx}.csv"
    print(f"  [+] Loading activations from Layer {layer_idx}...")
    act_df = pd.read_csv(act_file)
    print(f"  [+] Loading prompts for Principle {principle_idx}...")
    prompt_df = pd.read_csv(prompt_file)
    print(f"      -> Activations Shape: {act_df.shape} | Prompts Shape: {prompt_df.shape}")
    return act_df, prompt_df

def phase_1_and_2(activations, prompts_df, principle_idx, variance_threshold=20):
    print(f"  [+] Phase 1: Pruning dead features (Threshold: >= {variance_threshold} non-zero instances)...")
    
    act_matrix = activations.drop('Feature_ID', axis=1)
    alive_mask = (act_matrix > 0.0).sum(axis=1) >= variance_threshold
    alive_features = activations[alive_mask].copy()

    del act_matrix
    gc.collect()

    total_feats = activations.shape[0]
    survived_feats = alive_features.shape[0]
    print(f"      -> Survived: {survived_feats} / {total_feats} features ({(survived_feats/total_feats)*100:.2f}%)")

    plt.figure(figsize=(8, 5))
    sns.histplot((alive_features.drop('Feature_ID', axis=1) > 0.0).sum(axis=1), bins=50, kde=False)
    plt.axvline(variance_threshold, color='red', linestyle='dashed')
    plt.title(f"Feature Survival (Principle {principle_idx})")
    plt.savefig(f"{OUTPUT_PATH}survival_p{principle_idx}.png")
    plt.close()

    print(f"  [+] Phase 2: Identifying top 50 candidate features...")
    max_acts = alive_features.drop('Feature_ID', axis=1).max(axis=1)
    top_candidate_indices = max_acts.sort_values(ascending=False).head(50).index
    top_candidates = alive_features.loc[top_candidate_indices]

    feature_contexts = {}
    for i, (_, row) in enumerate(top_candidates.iterrows()):
        f_id = int(row['Feature_ID'])
        top_cols = row.drop('Feature_ID').sort_values(ascending=False).head(20)
        feature_contexts[str(f_id)] = [
            {"prompt": prompts_df.iloc[int(col.split('_')[1])]['Prompt'], "score": score, "idx": int(col.split('_')[1])}
            for col, score in top_cols.items() if score > 0
        ]
        
    print(f"      -> Contexts successfully extracted for top {len(feature_contexts)} candidates.")
    return feature_contexts, alive_features

def simulate_and_score(hypothesis, contexts, all_prompts_df, full_activations_row):
    if len(contexts) < 10: 
        return 0.0
    
    top_indices = [c['idx'] for c in contexts]
    all_nonzero = [i for i, val in enumerate(full_activations_row) if val > 0 and i not in top_indices]
    
    if len(all_nonzero) < 5: 
        return 0.0

    test_set = contexts[5:10] + [
        {"prompt": all_prompts_df.iloc[idx]['Prompt'], "score": full_activations_row[idx], "idx": idx} 
        for idx in np.random.choice(all_nonzero, 5, replace=False)
    ]
    np.random.shuffle(test_set)

    sim_prompt = f"Concept: '{hypothesis}'\nPredict activation score (0.0-10.0) for these 10 prompts. Output ONLY a valid JSON array of 10 floats.\n" + "\n".join([f"[{i+1}] {t['prompt']}" for i, t in enumerate(test_set)])

    response_text = safe_generate(sim_prompt)
    if not response_text:
        return 0.0

    try:
        match = re.search(r'\[(.*?)\]', response_text, re.DOTALL)
        if match:
            pred = json.loads(f"[{match.group(1)}]")
            if len(pred) == 10:
                actual_scores = [t['score'] for t in test_set]
                # Catch mathematical edge case where predictions are completely uniform (variance = 0)
                if np.std(pred) == 0 or np.std(actual_scores) == 0:
                    return 0.0
                return pearsonr(actual_scores, pred)[0]
        return 0.0
    except Exception: 
        return 0.0

def phase_3_and_4(contexts_dict, alive_features, all_prompts_df, p_idx):
    print("  [+] Phase 3 & 4: Beginning Autointerpretability (API calls starting)...")
    top_f_indices = alive_features.drop('Feature_ID', axis=1).max(axis=1).sort_values(ascending=False).head(10).index
    results = []

    for i, idx in enumerate(top_f_indices):
        f_id = str(int(alive_features.loc[idx, 'Feature_ID']))
        print(f"      -> Processing Feature ID: {f_id} ({i+1}/10)")
        
        if f_id not in contexts_dict:
            continue
            
        examples = "\n\n".join([f"Score: {c['score']:.2f}\nPrompt: {c['prompt']}" for c in contexts_dict[f_id][:5]])
        prompt_text = f"Look at these activations.\n{examples}\nWhat abstract semantic concept does this feature represent? 1-2 sentences."
        
        hypothesis = safe_generate(prompt_text)
        
        if hypothesis:
            print(f"         [API] Hypothesis generated: {hypothesis[:75]}...")
            score = simulate_and_score(hypothesis, contexts_dict[f_id], all_prompts_df, alive_features.loc[idx].drop('Feature_ID').values)
            # Handle NaN returns from scipy if correlation fails
            if np.isnan(score): score = 0.0 
            print(f"         [API] Simulation complete. Pearson Score: {score:.4f}")
            results.append({"Feature_ID": f_id, "Hypothesis": hypothesis, "Score": score})
        else:
            print(f"         [!] Skipped Feature {f_id} due to API failure.")
            continue
            
        time.sleep(2) 

    res_df = pd.DataFrame(results)
    if not res_df.empty:
        plt.figure(figsize=(10, 6))
        sns.barplot(x=res_df['Feature_ID'].astype(str), y=res_df['Score'])
        plt.title(f"Autointerpretability Scores (Principle {p_idx})")
        plt.savefig(f"{OUTPUT_PATH}scores_p{p_idx}.png")
        plt.close()
        print("  [+] Phase 4 Complete. Scores plot saved.")
    return res_df

def phase_5_cross_validation(top_features, source_p_idx):
    if not top_features: 
        print("  [!] No top features to cross-validate. Skipping Phase 5.")
        return
    
    print(f"  [+] Phase 5: Cross-Principle Validation for {len(top_features)} abstract features...")
    heat_data = np.zeros((len(top_features), 12))
    
    # Ensure targets are integers for strict comparison
    target_ids = [int(f) for f in top_features]

    for p_idx, l_idx in LAYER_MAPPING.items():
        act_file = f"{BASE_ACT_PATH}Raw_Activations_Principle{p_idx}_Layer_{l_idx}.csv"
        try:
            # Bulletproof chunking logic: Never assume Row Index = Feature_ID
            found_count = 0
            for chunk in pd.read_csv(act_file, chunksize=10000):
                mask = chunk['Feature_ID'].isin(target_ids)
                if mask.any():
                    matched = chunk[mask]
                    for _, row in matched.iterrows():
                        f_id = int(row['Feature_ID'])
                        row_idx = target_ids.index(f_id)
                        heat_data[row_idx, p_idx-1] = row.drop('Feature_ID').mean()
                        found_count += 1
                if found_count == len(target_ids):
                    break # Optimization: Stop parsing the 1.5GB CSV once we have our 5 features
        except Exception as e: 
            print(f"      [!] Error processing Phase 5 file P{p_idx}: {e}")
            pass
    
    print("      -> Heatmap data matrix built successfully.")
    plt.figure(figsize=(12, 8))
    sns.heatmap(heat_data, xticklabels=[f"P{i}" for i in range(1, 13)], yticklabels=top_features, cmap="viridis", annot=True, fmt=".2f")
    plt.title(f"Cross-Principle Validation: Features from Principle {source_p_idx}")
    plt.savefig(f"{OUTPUT_PATH}heatmap_p{source_p_idx}.png")
    plt.close()
    print("  [+] Phase 5 Complete. Heatmap saved.")


 

In [5]:
LAYER_MAPPING1 = {
    4: 4, 5: 16, 6: 19,
    7: 28, 8: 3, 9: 13, 10: 11, 11: 9, 12: 13
}
# 3. Master Loop Execution
# ==========================================
print("=========================================================")
print("INITIATING SAE MECHANISTIC INTERPRETABILITY PIPELINE")
print("=========================================================\n")

for target_principle, target_layer in LAYER_MAPPING1.items():
    print(f"\n=========================================================")
    print(f"=== Starting Pipeline for Principle {target_principle} (Layer {target_layer}) ===")
    print(f"=========================================================")

    acts, prompts = load_data(target_principle, target_layer)
    contexts, alive = phase_1_and_2(acts, prompts, target_principle)

    print(f"  [+] Exporting Top Contexts dictionary to JSON...")
    with open(f"{OUTPUT_PATH}Top_Prompts_P{target_principle}.json", "w") as f:
        json.dump(contexts, f, indent=4)

    results_df = phase_3_and_4(contexts, alive, prompts, target_principle)
    
    print(f"  [+] Exporting Interpretability Dictionary to CSV...")
    results_df.to_csv(f"{OUTPUT_PATH}Interpretability_Dict_P{target_principle}.csv", index=False)

    if not results_df.empty:
        print("  [+] Isolating top 5 abstract features for cross-validation...")
        top_abstract = results_df.sort_values(by="Score", ascending=False).head(5)['Feature_ID'].tolist()
        
        print("  [+] Flushing memory (Garbage Collection)...")
        del acts, prompts, contexts, alive
        gc.collect()

        phase_5_cross_validation(top_abstract, target_principle)
    
    print(f"\n=== Principle {target_principle} Complete. All Artifacts Exported to /kaggle/working/ ===")

INITIATING SAE MECHANISTIC INTERPRETABILITY PIPELINE


=== Starting Pipeline for Principle 4 (Layer 4) ===
  [+] Loading activations from Layer 4...
  [+] Loading prompts for Principle 4...
      -> Activations Shape: (131072, 1794) | Prompts Shape: (1793, 3)
  [+] Phase 1: Pruning dead features (Threshold: >= 20 non-zero instances)...
      -> Survived: 23514 / 131072 features (17.94%)
  [+] Phase 2: Identifying top 50 candidate features...
      -> Contexts successfully extracted for top 50 candidates.
  [+] Exporting Top Contexts dictionary to JSON...
  [+] Phase 3 & 4: Beginning Autointerpretability (API calls starting)...
      -> Processing Feature ID: 84288 (1/10)
         [!] Rate limit hit. Retrying in 5s... (Attempt 1/6)
         [!] Rate limit hit. Retrying in 10s... (Attempt 2/6)
         [!] Rate limit hit. Retrying in 20s... (Attempt 3/6)
         [!] Rate limit hit. Retrying in 40s... (Attempt 4/6)
         [!] Rate limit hit. Retrying in 80s... (Attempt 5/6)


KeyboardInterrupt: 